### The following code is related to Fig.2b (left and bottom), Supplementary Fig.9c–e, Fig.2h, Fig.3g, Supplementary Fig.13, and Supplementary Fig.14.

In [ ]:
library(tibble)
library(stringr)
library(reshape2)
library(tidyr)
library(furrr)
library(future)
library(dplyr)
library(Seurat)
library(ggplot2)
options(future.globals.maxSize = 20 * 1024^3) 

In [ ]:
parallel::detectCores()

### data reading

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/')
seurat_obj <- readRDS('./result_zxs/scdata_filter.rds')

In [ ]:
seurat_obj <- subset(seurat_obj,subset = batch %in% c('normal2','Tatin'))

In [ ]:
seurat_obj$batch %>% table()

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "LogNormalize", margin =2)
seurat_obj

In [ ]:
meta_data <- seurat_obj@meta.data %>% 
    mutate(
        batch = ifelse(batch == 'Tatin',yes = 'Tatin',no = 'Normal'),
        function_type = case_when(
            sgRNA_type %in% c("SREBF2", "HMGCR", "SQLE", "INSIG1") ~ 'chol_synthesis',
            sgRNA_type %in% c("LDLR", "NPC1L1", "NPC1") ~ 'chol_uptake',
            sgRNA_type %in% c("APOB", "MTTP", "ABCA1", "ABCG1", "ABCG5", "ABCG8", "NR1H3") ~ 'chol_efflux',
            sgRNA_type %in% c("SOAT1") ~ 'chol_esterification',
            sgRNA_type %in% c("AAVS",'NT') ~ 'NT',
            TRUE ~ sgRNA_type
        ),
        sgRNA_type_use = ifelse(sgRNA_type %in% c("AAVS",'NT'),yes = 'NT',no = sgRNA_type),
        sgRNA_type_use = factor(sgRNA_type_use,levels = c(
            "NT","SREBF2","HMGCR","SQLE","INSIG1","LDLR","NPC1L1","NPC1","APOB",
            "MTTP","ABCA1","ABCG1","ABCG5","ABCG8","NR1H3","SOAT1"
        ))
    )
seurat_obj@meta.data <- meta_data
meta_data$batch %>% table()
meta_data$sgRNA_type_use %>% unique()

In [ ]:
color_use <- c('#96B6D8','#97C8AF')

In [ ]:
dir.create('/mnt/data/khm_scRNA/huh7_zxs/result_figs',showWarnings = FALSE)
getwd()

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
seurat_obj <- FindVariableFeatures(seurat_obj, assay = "RNA")
seurat_obj <- ScaleData(seurat_obj, assay = "RNA")
seurat_obj <- RunPCA(seurat_obj, assay = "RNA", reduction.name = "pca_rna")

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
# we will use all ADT features for dimensional reduction
# we set a dimensional reduction name to avoid overwriting the 
VariableFeatures(seurat_obj) <- rownames(seurat_obj[["Protein"]])
seurat_obj <- ScaleData(seurat_obj, assay = "Protein")
seurat_obj <- RunPCA(seurat_obj, assay = "Protein", reduction.name = "pca_adt",npcs = 30)

In [ ]:
seurat_obj

In [ ]:
# WNN
seurat_obj <- FindMultiModalNeighbors(seurat_obj, reduction.list = list("pca_rna", "pca_adt"), dims.list = list(1:10, 1:8))

seurat_obj <- RunUMAP(seurat_obj, nn.name = "weighted.nn", reduction.name = "wnn.umap")
seurat_obj <- FindClusters(seurat_obj, graph.name = "wsnn")

In [ ]:
seurat_obj@meta.data %>% colnames()

In [ ]:
seurat_obj$sgRNA_type %>% unique()

### Fig.2 b left

In [ ]:
options(repr.plot.width = 12,repr.plot.height = 9)
p <- DimPlot(
    seurat_obj, reduction = 'wnn.umap',group.by = 'batch',
    label = FALSE, repel = TRUE, pt.size = 2,alpha = 0.9) + 
    scale_color_manual(values = color_use,labels = c('Normal', 'Tatin')) +
    theme(
        plot.title = element_blank(),
        legend.text = element_text(size = 15),
        axis.title = element_text(size = 18)
    )
p
ggsave(plot = p,filename = './result_figs/Umap_batch.pdf',width = 12,height = 9)

### Fig.2 b bottom

In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(dplyr)
# 提取数据
DefaultAssay(seurat_obj) <- 'Protein'
data_plot <- FetchData(seurat_obj, vars = c("ALOD4-pAbO",'batch'))
options(repr.plot.width = 8,repr.plot.height = 9)
p <- ggplot(data_plot, aes(x = batch, y = `ALOD4-pAbO`, fill = batch)) +
  geom_violin(position = position_dodge(width = 0.9), trim = FALSE) +
  geom_boxplot(width = 0.2, position = position_dodge(width = 0.9), color = 'white',outlier.shape = NA,show.legend = FALSE) +
  scale_fill_manual(values = color_use,name = 'Batch') +
  labs(x = 'Group',y = 'ALOD4(CLR-transformed ADT counts)') +
  stat_compare_means(
    comparisons = list(c("Normal", "Tatin")),
    label = "p.signif",
    method = "wilcox.test",size = 8
    # position = position_dodge(width = 0.9)
  ) +
  theme_classic(base_size = 20)
p
ggsave(plot = p,filename = './result_figs/Vlnplot_ALOD4_NT.pdf',width = 8,height = 9)

### Supplementary Fig.9 d

In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(dplyr)
# 提取数据
data_plot <- FetchData(seurat_obj, vars = c("ALOD4-pAbO", "sgRNA_type_use", "batch")) %>% 
    mutate(
        # sgRNA_type_use = factor(sgRNA_type_use,levels = c(
            
        # ))
    )
options(repr.plot.width = 24,repr.plot.height = 9)
p <- ggplot(data_plot, aes(x = sgRNA_type_use, y = `ALOD4-pAbO`, fill = batch)) +
  geom_violin(position = position_dodge(width = 0.9), trim = TRUE) +
  geom_boxplot(width = 0.2, position = position_dodge(width = 0.9),color = 'white', outlier.shape = NA) +
  scale_fill_manual(values = color_use,name = 'Group') +
  labs(x = 'gRNA Types',y = 'ALOD4(CLR-transformed ADT counts)') +
  stat_compare_means(
    # aes(group = batch,fill = batch),
    # comparisons = list(c("normal2", "Tatin")),
    label = "p.signif",
    method = "wilcox.test",size = 8
    # position = position_dodge(width = 0.9)
  ) +
  theme_classic(base_size = 20)
p
ggsave(plot = p,filename = './result_figs/Vlnplot_ALOD4_all_sgRNA.pdf',width = 24,height = 9)

In [ ]:
VlnPlot(object = seurat_obj,features = c("ALOD4-pAbO","SOAT1-pAbO",'LDLR-pAbO'),group.by = 'sgRNA_type',pt.size = 0) &
    scale_fill_manual(values = paletteer::paletteer_d("wesanderson::Royal1", 17, type = "continuous"))

In [ ]:
# 分别找变量特征、PCA
DefaultAssay(seurat_obj) <- 'RNA'
seurat_obj <- FindVariableFeatures(seurat_obj, assay = "RNA")
seurat_obj <- ScaleData(seurat_obj, assay = "RNA")
seurat_obj <- RunPCA(seurat_obj, assay = "RNA", reduction.name = "pca_rna")

In [ ]:
seurat_obj$

In [ ]:
ElbowPlot(seurat_obj,reduction = "pca_rna", ndims = 50)

In [ ]:
seurat_obj <- RunUMAP(seurat_obj, assay = "RNA", reduction.name = "umap_rna",dims = 1:30)
seurat_obj

In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(dplyr)
library(purrr)

In [ ]:
seurat_obj@meta.data %>% head()

In [ ]:
table(seurat_obj$nCount_Protein>300)

In [ ]:
data_plot <- FetchData(seurat_obj, vars = c("nCount_Protein", "nFeature_Protein", "batch","sgRNA_identity")) %>% 
    mutate(
        sgRNA_identity = ifelse(sgRNA_identity %in% c('AAVS','NT1','NT2'), yes = 'NT', no = sgRNA_identity),
        sgRNA_identity = factor(sgRNA_identity,levels = sgRNA_identity %>% unique() %>% sort())
    ) %>% 
    pivot_longer(cols = c("nCount_Protein", "nFeature_Protein"),names_to = 'facet_use',values_to = 'Number') %>% 
    filter(!(facet_use == 'nCount_Protein' & Number > 300)) %>% 
    mutate(facet_use = case_when(
        facet_use == 'nFeature_Protein' ~ 'Number of detected antibodies',
        facet_use == 'nCount_Protein' ~ 'Total antibody UMI count per cell',
    ))
data_plot %>% head()

In [ ]:
options(repr.plot.width = 42,repr.plot.height = 18)
p <- ggplot(data_plot, aes(x = sgRNA_identity, y = Number, fill = batch)) +
  geom_violin(position = position_dodge(width = 0.9), trim = TRUE) +
  geom_boxplot(width = 0.2, position = position_dodge(width = 0.9),color = 'white', outlier.shape = NA) +
  facet_wrap(~ facet_use,ncol = 1,,scale = 'free',strip.position = 'top') +
  scale_fill_manual(values = color_use,name = 'Group') +
  labs(x = 'gRNA Types',y = 'ALOD4(CLR-transformed ADT counts)') +
  stat_compare_means(
    label = "p.signif",
    method = "wilcox.test",size = 8
  ) +
  theme_classic(base_size = 20) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.title = element_text(size = 24,angle = 0),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    axis.text.x = element_text(size = 20,angle = 90,hjust = 1,vjust = 0.5),
    axis.title = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )
p


In [ ]:
data_plot %>% head()

In [ ]:
data_plot %>% 
    group_by(facet_use,batch) %>% 
    summarise(median(Number))
data_plot %>% 
    group_by(facet_use) %>% 
    summarise(median(Number))

### Supplementary Fig.9 c

In [ ]:
library(introdataviz)
library(ggpubr)
options(repr.plot.width = 12,repr.plot.height = 8)
p <- ggplot(data_plot %>% mutate(celltype = 'one'), aes(x = celltype, y = Number, fill = celltype)) + 
  geom_violin(adjust = 3,alpha = 0.75,scale = 'area', trim = F,color = NA,width = 0.6) + 
  geom_boxplot(width = 0.15, alpha = .6, position = position_dodge(width = 0.2), outlier.shape = NA,show.legend = FALSE,color = 'white') +
  facet_wrap(~ facet_use,ncol = 2,,scale = 'free',strip.position = 'top') +
  ylab("Value") + xlab(NULL) +
  scale_fill_manual(name = 'Group',values = c('#96B6D8','#97C8AF'),) +
  theme_classic(base_size = 20) +
  guides(fill = guide_legend(title.position = "top", title.theme = element_text(angle = 0))) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.position = 'none',
    legend.title = element_text(size = 24,angle = 0),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    # axis.text.x = element_text(size = 20,angle = 90,hjust = 1,vjust = 0.5),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )
p


In [ ]:
seurat_obj@meta.data %>% colnames()

In [ ]:
data_plot <- FetchData(seurat_obj, vars = c("nCount_RNA", "nFeature_RNA", "batch","sgRNA_identity")) %>% 
    mutate(
        sgRNA_identity = ifelse(sgRNA_identity %in% c('AAVS','NT1','NT2'), yes = 'NT', no = sgRNA_identity),
        sgRNA_identity = factor(sgRNA_identity,levels = sgRNA_identity %>% unique() %>% sort())
    ) %>% 
    pivot_longer(cols = c("nCount_RNA", "nFeature_RNA"),names_to = 'facet_use',values_to = 'Number') %>% 
    mutate(facet_use = case_when(
        facet_use == 'nFeature_RNA' ~ 'Number of detected mRNA',
        facet_use == 'nCount_RNA' ~ 'Total mRNA UMI count per cell',
    ))
data_plot %>% head()

In [ ]:
data_plot %>% 
    group_by(facet_use,batch) %>% 
    summarise(median(Number))
data_plot %>% 
    group_by(facet_use) %>% 
    summarise(median(Number))

In [ ]:
library(introdataviz)
library(ggpubr)
options(repr.plot.width = 12,repr.plot.height = 8)
p <- ggplot(data_plot %>% mutate(celltype = 'one'), aes(x = celltype, y = Number, fill = celltype)) + 
  geom_violin(adjust = 3,alpha = 0.75,scale = 'area', trim = F,color = NA,width = 0.6) + 
  geom_boxplot(width = 0.15, alpha = .6, position = position_dodge(width = 0.2), outlier.shape = NA,show.legend = FALSE,color = 'white') +
  facet_wrap(~ facet_use,ncol = 2,,scale = 'free',strip.position = 'top') +
  ylab("Value") + xlab(NULL) +
  scale_fill_manual(name = 'Group',values = c('#96B6D8','#97C8AF'),) +
  theme_classic(base_size = 20) +
  guides(fill = guide_legend(title.position = "top", title.theme = element_text(angle = 0))) +
  theme(
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.position = 'none',
    legend.title = element_text(size = 24,angle = 0),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.2, "cm"),
    legend.key.width = unit(1.2, "cm"),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )
p
ggsave(plot = p,filename = './result_figs/nCount_nFeature_mRNA.pdf',width = 12,height = 8)

In [ ]:
seurat_use <- subset(seurat_obj,subset = sgRNA_identity %in% c('AAVS','NT1','NT2'))
DefaultAssay(seurat_use) <- 'RNA'
seurat_use$sgRNA_identity %>% table()
seurat_use$batch %>% table()
seurat_use

In [ ]:
diffgene <- FindMarkers(
    object = seurat_use,
    slot = 'data',
    group.by = 'batch',
    ident.1	= 'Tatin',
    min.pct = 0,
    random.seed	= 1234,
    logfc.threshold = 0,
    only.pos = FALSE
) %>% 
    mutate(
        group = case_when(
            (p_val<0.05) & (avg_log2FC > 0.5) ~ 'Up in Tatin',
            (p_val<0.05) & (avg_log2FC < -0.5) ~ 'Down in Tatin',
            TRUE ~ 'Stable'
        )
    )
diffgene$group %>% table()
diffgene %>% head()

In [ ]:
gene_select <- c("HMGCR", "HMGCS1", "MVK", "MVD", "FDPS", "FDFT1", "SQLE", 
           "CYP51A1", "TM7SF2", "NSDHL", "MSMO1", "EBP", "SREBF2", "LDLR")
all(gene_select %in% rownames(diffgene))

In [ ]:
diffgene_label <- diffgene %>% 
    rownames_to_column('Genes') %>% 
    filter(Genes %in% gene_select)
diffgene_label

### Supplementary Fig.9 e

In [ ]:
library(ggrepel)
cut_off_logFC <- 0.5
cut_off_pvalue <- 0.05
options(repr.plot.width = 14,repr.plot.height = 12)
ggplot(diffgene, aes(x = avg_log2FC, y = -log10(p_val), colour=group)) +
    geom_point(alpha=0.9, size=3.5) +
    geom_point(
        data = diffgene_label,
        mapping = aes(x = avg_log2FC, y = -log10(p_val),fill = group),
        color = 'black',shape = 21,alpha=0.9, size=3.5,show.legend = FALSE) +
    geom_text_repel(
        data = diffgene_label,
        mapping = aes(x = avg_log2FC, y = -log10(p_val), label = Genes),
        color = 'black',size = 4,show.legend = FALSE,nudge_y = 1
    ) +
    scale_color_manual(name = 'Group',values=c("#9EB8D6", "#d2dae2","#9EC8B2"))+
    geom_vline(
        xintercept=c(-cut_off_logFC,cut_off_logFC),
        lty=4, col="black", lwd=0.8) +
    geom_hline(yintercept = -log10(cut_off_pvalue),
        lty=4, col="black", lwd=0.8) +
    labs(x="log2(fold change)", y="-log10 (p-value)")+
    theme_bw()+
    theme(
        legend.position="right",
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        axis.title = element_text(size = 24)
    )
ggsave(filename = './result_figs/NTsg_RNA_expression.pdf',width = 14,height = 12,limitsize = FALSE)

In [ ]:
library(ggplot2)
library(ggpmisc) 

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
sgRNA_identity_select <- seurat_obj$sgRNA_identity %>%  unique() %>% sort() %>% .[!grepl('NT|AAVS',.)]
sgRNA_identity_select

In [ ]:
rownames(seurat_obj)

In [ ]:
df_res <- lapply(sgRNA_identity_select,function(sgRNA_identity_i){
    print(sgRNA_identity_i)
    DefaultAssay(seurat_obj) <- 'RNA'
    seurat_use <- subset(seurat_obj,subset = sgRNA_identity %in% c('AAVS','NT1','NT2',sgRNA_identity_i))
    meta_data_use <- seurat_use@meta.data %>% 
        mutate(
            sgRNA_identity = case_when(
                sgRNA_identity %in% c('AAVS','NT1','NT2') ~ 'NT',
                TRUE ~ sgRNA_identity 
        ))
    seurat_use@meta.data <- meta_data_use
    diffgene <- FindMarkers(
        object = seurat_use,
        slot = 'data',
        group.by = 'sgRNA_identity',
        ident.1	= sgRNA_identity_i,
        min.pct = 0,
        random.seed	= 1234,
        logfc.threshold = 0,
        only.pos = FALSE
    ) %>% 
        rownames_to_column('Genes') %>% 
        mutate(
            group = case_when(
                (p_val<0.05) & (avg_log2FC > 0.5) ~ paste('Up in ',sgRNA_identity_i),
                (p_val<0.05) & (avg_log2FC < -0.5) ~ paste('Down in ',sgRNA_identity_i),
                TRUE ~ 'Stable'
            ),
            sgRNA_identity = sgRNA_identity_i
        )
    return(diffgene)
}) %>% do.call(rbind,.)
df_res %>% head()

In [ ]:
df_res <- df_res %>% 
    filter(!grepl('^ENSG',Genes))

In [ ]:
df_res$sgRNA_identity %>% table()

In [ ]:
get_two_sg_fc_matrix <- function(target, sg1, sg2, df_res) {
  id1 <- paste0(target, "-sg", sg1)
  id2 <- paste0(target, "-sg", sg2)
  # 筛选
  sub_df <- df_res %>% 
    filter(sgRNA_identity %in% c(id1, id2)) %>% 
    pivot_wider(
      id_cols = Genes,
      names_from = sgRNA_identity,
      values_from = c(avg_log2FC, p_val)
    ) %>% 
    filter((.[[4]] < 0.01  & abs(.[[2]]) > 0.05) | (.[[5]] < 0.01  & abs(.[[3]]) > 0.05)) %>% 
    select(-c(names(.)[4],names(.)[5])) %>% 
    rename_all(~c('Genes','first','second')) %>% 
    mutate(facet_use = paste("sg",sg1,' vs ',"sg",sg2,sep = ''))
  return(sub_df)
}

In [ ]:
df_res %>% head()

In [ ]:
data_plot <- list(
    get_two_sg_fc_matrix("LDLR", 1, 2, df_res),
    get_two_sg_fc_matrix("LDLR", 1, 3, df_res),
    get_two_sg_fc_matrix("LDLR", 2, 3, df_res)
) %>% do.call(rbind,.)
data_plot$facet_use %>% table()
data_plot %>% head()

In [ ]:
data_plot %>% 
    group_by(first,facet_use) %>%
    filter(n() > 1) %>% 
    arrange(desc(first))

### Fig.2 h

In [ ]:
options(repr.plot.width = 24,repr.plot.height = 7)
ggplot(data = data_plot,aes(x = first,y = second)) +
    geom_point(color = '#D281A4',alpha = 0.4) +##D281A4
    geom_smooth(
        method = "lm",formula = y ~ x, se = TRUE, 
        fill = "#c9d6df",alpha = 0.7,color = "#355c7d"
    ) +
    stat_correlation(
        method      = "spearman",
        output.type = "numeric",
        r.digits    = 3,p.digits    = 3,exact = FALSE,
        aes(label = sprintf(
          "Spearman corr coef = %.3f\nP-value = %.3f",
          after_stat(rho),             # 相关系数
          after_stat(p.value)          # P 值
        )),
        size = 6
      ) +
    facet_wrap(~facet_use,ncol = 3,strip.position = 'left',scales = 'free') +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.2, "cm"),
        legend.key.width = unit(1.2, "cm"),
        # axis.text.x = element_blank(),
        axis.title.y = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
    )

In [ ]:
p <- ggplot(data_plot, aes(x = first, y = second)) +
  stat_correlation(
    method      = "spearman",
    output.type = "numeric",exact = FALSE
  )

layer_df <- ggplot_build(p)$data[[1]]
names(layer_df)
layer_df %>% head()

In [ ]:
sgRNA_identity_use <- sgRNA_identity_select %>% str_remove('-sg1|-sg2|-sg3') %>% sort() %>% unique()
sgRNA_identity_use

In [ ]:
for(sgRNA_identity_i in sgRNA_identity_use){
    print(sgRNA_identity_i)
    data_plot <- list(
        get_two_sg_fc_matrix(sgRNA_identity_i, 1, 2, df_res),
        get_two_sg_fc_matrix(sgRNA_identity_i, 1, 3, df_res),
        get_two_sg_fc_matrix(sgRNA_identity_i, 2, 3, df_res)
    ) %>% do.call(rbind,.)
    data_plot$facet_use %>% table()
    data_plot %>% head()
    options(repr.plot.width = 24,repr.plot.height = 7)
    p <- ggplot(data = data_plot,aes(x = first,y = second)) +
        geom_point(color = '#D281A4') +
        geom_smooth(
            method = "lm",formula = y ~ x, se = TRUE, 
            fill = "#c9d6df",alpha = 0.7,color = "#355c7d"
        ) +
        stat_correlation(
            method = "spearman",
            output.type = "numeric",
            r.digits = 3,p.digits    = 3,exact = FALSE,
            aes(label = sprintf(
              "Spearman corr coef = %.3f\nP-value = %.3f",
              after_stat(rho),             
              after_stat(p.value)         
            )),
            size = 6
          ) +
        facet_wrap(~facet_use,ncol = 3,strip.position = 'left',scales = 'free') +
        theme_classic() +
        theme(
            plot.title = element_text(hjust = 0.5,size = 32),
            legend.title = element_text(size = 24),
            legend.text = element_text(size = 20),
            legend.key.height = unit(1.2, "cm"),
            legend.key.width = unit(1.2, "cm"),
            axis.title.y = element_blank(),
            strip.background = element_blank(),
            strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
            strip.placement = 'outside'
        )
    file_name = paste('./result_figs/DEG_correlations_within_sgRNA_',sgRNA_identity_i,'.pdf',sep = '')
    print(file_name)
    ggsave(filename = file_name,plot = p,width = 24,height = 7,limitsize = FALSE)
}

### Fig.3 g

In [ ]:
seurat_use <- seurat_obj
seurat_use$sgRNA_identity <- ifelse(seurat_use$sgRNA_identity %in% c('NT1','NT2','AAVS'),yes = 'NT',no = seurat_use$sgRNA_identity)

In [ ]:
Protein_select <- c("ALOD4","PDL1","pGSK3b","MYC")
Protein_select <- rownames(seurat_use) %>% .[grepl(paste(Protein_select,collapse = '|'),.,ignore.case = TRUE)]
Protein_select
sgRNA_identity_select <- c("LDLR-sg1", "LDLR-sg2", "LDLR-sg3")

In [ ]:
DefaultAssay(seurat_use) <- 'Protein'
data_plot_sgRNA <- FetchData(seurat_use, vars = c(Protein_select, "sgRNA_identity", "batch",'sgRNA_type')) %>% 
    mutate(color_use = sgRNA_identity %>% str_split('-') %>% purrr::map_chr(~ .[2])) %>% 
    pivot_longer(cols = Protein_select,names_to = 'Genes',values_to = 'Expression') %>% 
    filter(
        (batch == 'Tatin' & sgRNA_identity %in% sgRNA_identity_select)
    ) %>% 
    mutate(Identity_use = paste(sgRNA_identity,Genes,batch,sep = '_'),group = paste(Genes,batch,sep = '_'))
data_plot_sgRNA %>% dim()
data_plot_sgRNA %>% head()

In [ ]:
data_plot_sub <- FetchData(seurat_use, vars = c(Protein_select, "sgRNA_identity", "batch",'sgRNA_type')) %>% 
    mutate(color_use = sgRNA_identity %>% str_split('-') %>% purrr::map_chr(~ .[2])) %>% 
    pivot_longer(cols = Protein_select,names_to = 'Genes',values_to = 'Expression') %>% 
    filter(is.na(color_use),batch == 'Tatin') %>% 
    mutate(color_use = 'NT') %>% 
    mutate(Identity_use = paste(sgRNA_identity,Genes,batch,sep = '_'),group = paste(Genes,batch,sep = '_'))
data_plot_sub$sgRNA_identity %>% unique()
data_plot_sub$batch %>% unique()
data_plot_sub %>% dim()
data_plot_sub %>% head()

In [ ]:
data_plot_sgRNA$sgRNA_identity %>% unique() %>% length()
data_plot_sgRNA$sgRNA_identity %>% unique() %>% sort()

In [ ]:
data_plot_sgRNA$Identity_use %>% unique()

In [ ]:
data_plot_sgRNA$sgRNA_type %>% unique()

In [ ]:
data_plot <- lapply(data_plot_sgRNA$Genes %>% unique(),FUN = function(x){
    print(x)
    tmp  <- data_plot_sub %>% 
        filter(Genes == x) %>% 
        mutate(sgRNA_type = x)
    return(tmp)
}) %>% do.call(rbind,.) %>% 
    rbind(data_plot_sgRNA) %>% 
    mutate(
        color_use = factor(color_use,levels = c('NT','sg1','sg2','sg3'))
    )
data_plot %>% dim()
data_plot %>% head()

In [ ]:
data_plot$color_use %>% unique()

In [ ]:
nt_lines <- data_plot %>%
  filter(batch == "Tatin", color_use == "NT") %>%
  group_by(Genes) %>%
  summarise(mean_expr = mean(Expression, na.rm = TRUE), .groups = "drop") %>%
  mutate(x = 0.1, xend = 4.8)
nt_lines %>% head()

In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(dplyr)
options(repr.plot.width = 16,repr.plot.height = 7)
p <- ggplot(data_plot %>% filter(batch == 'Tatin'), aes(x = color_use, y = Expression)) +
  geom_violin(aes(fill = color_use),adjust = 3,position = position_dodge(width = 0.9), trim = FALSE) +
  geom_boxplot(alpha = 0.1,width = 0.5, fatten = 0,position = position_dodge(width = 0.9), outlier.shape = 16,show.legend = FALSE) +
  stat_summary(fun = mean, geom = "errorbar", aes(ymax = ..y.., ymin = ..y..),
               width = 0.5, size = 1, linetype = "solid",color = 'black') +
  geom_segment(data = nt_lines,
               aes(x = x, xend = xend, y = mean_expr, yend = mean_expr),
               linetype = "longdash",inherit.aes = FALSE, color = "grey98", linewidth = 0.8) +
  stat_compare_means(
    inherit.aes = FALSE,             
    mapping = aes(
        x     = color_use,              
        y     = Expression,             
        group = color_use              
    ),
    comparisons = list(
        c('NT','sg1'),c('NT','sg2'),c('NT','sg3')
    ),
    hide.ns = TRUE,
    label = "p.signif",
    method = "wilcox.test",size = 8
  ) +
  scale_fill_manual(name = 'sgRNA Type',values = c("#F4A88E","#ABD1EB","#6BABD9","#739DC7")) +
  scale_color_manual(name = 'sgRNA Type',values = c("#F4A88E","#ABD1EB","#6BABD9","#739DC7")) +
  theme_classic(base_size = 20) +
  facet_wrap(~ Genes,nrow = 1,strip.position = 'left',scales = 'free') +
  guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 90))) +
  theme(
      panel.border     = element_blank(),  
      plot.title       = element_text(hjust = 0.5, size = 32),
      legend.title     = element_text(size = 24),
      legend.text      = element_text(size = 20),
      legend.key.height = unit(1.2, "cm"),
      legend.key.width  = unit(1.2, "cm"),
      axis.text.x      = element_blank(),
      axis.text.y      = element_blank(),
      axis.title       = element_blank(),
      strip.background = element_blank(),                
      strip.text       = element_text(size = 20, face = 'italic', vjust = 0.7),
      strip.placement  = 'outside'                       
    )
p
ggsave(plot = p,filename = './result_figs/Boxplot_protein-NT_Tatin.pdf',width = 16,height = 7,limitsize = FALSE)

In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(dplyr)
options(repr.plot.width = 16,repr.plot.height = 7)
p <- ggplot(data_plot %>% filter(batch == 'Tatin'), aes(x = color_use, y = Expression)) +
  geom_violin(aes(fill = color_use),adjust = 3,position = position_dodge(width = 0.9), trim = FALSE) +
  geom_boxplot(alpha = 0.1,width = 0.5, position = position_dodge(width = 0.9), outlier.shape = 16,show.legend = FALSE) +
  stat_summary(fun = mean, geom = "errorbar", aes(ymax = ..y.., ymin = ..y..),
               width = 0.5, size = 1, linetype = "solid",color = 'red') +
  geom_segment(data = nt_lines,
               aes(x = x, xend = xend, y = mean_expr, yend = mean_expr),
               linetype = "longdash",inherit.aes = FALSE, color = "grey98", linewidth = 0.8) +
  stat_compare_means(
    inherit.aes = FALSE,              
    mapping = aes(
        x     = color_use,               
        y     = Expression,           
        group = color_use              
    ),
    comparisons = list(
        c('NT','sg1'),c('NT','sg2'),c('NT','sg3')
    ),
    hide.ns = TRUE,
    label = "p.signif",
    method = "wilcox.test",size = 8
  ) +
  scale_fill_manual(name = 'sgRNA Type',values = c("#F4A88E","#ABD1EB","#6BABD9","#739DC7")) +
  scale_color_manual(name = 'sgRNA Type',values = c("#F4A88E","#ABD1EB","#6BABD9","#739DC7")) +
  theme_classic(base_size = 20) +
  facet_wrap(~ Genes,nrow = 1,strip.position = 'left',scales = 'free') +
  guides(fill = guide_legend(title.position = "left", title.theme = element_text(angle = 90))) +
  theme(
      panel.border     = element_blank(),  
      plot.title       = element_text(hjust = 0.5, size = 32),
      legend.title     = element_text(size = 24),
      legend.text      = element_text(size = 20),
      legend.key.height = unit(1.2, "cm"),
      legend.key.width  = unit(1.2, "cm"),
      axis.text.x      = element_blank(),
      axis.text.y      = element_blank(),
      axis.title       = element_blank(),
      strip.background = element_blank(),                
      strip.text       = element_text(size = 20, face = 'italic', vjust = 0.7),
      strip.placement  = 'outside'                       
    )
p


## Supplementary Fig. 13 and Supplementary Fig. 14

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj_use <- subset(seurat_obj,subset = batch == 'Tatin')
meta_data <- seurat_obj_use@meta.data %>% 
    mutate(
        sgRNA_identity = case_when(
            sgRNA_identity %in% c("AAVS",'NT1','NT2') ~ 'NT',
            TRUE ~ sgRNA_identity
        )
    ) %>% 
    filter(
        sgRNA_identity %in% c('LDLR-sg1','LDLR-sg2','LDLR-sg3','SOAT1-sg1','SOAT1-sg2','SOAT1-sg3','NT')
    )
meta_data$sgRNA_identity %>% unique()
meta_data %>% head()
seurat_obj_use <- subset(seurat_obj_use,cells = rownames(meta_data))
seurat_obj_use@meta.data <- meta_data

In [ ]:
Idents(seurat_obj_use) <- seurat_obj_use$sgRNA_type
diff_gene <- FindMarkers(
    object = seurat_obj_use,
    ident.1 = 'LDLR-sg1',ident.2 = 'NT',
    group.by = 'sgRNA_identity',
    logfc.threshold = 0.1,
    test.use = "wilcox",
    only.pos = FALSE
)

In [ ]:
getwd()

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/result_figs/KEGG pathway/')

In [ ]:
sgRNA_identity_list <- c('LDLR-sg1','LDLR-sg2','LDLR-sg3','SOAT1-sg1','SOAT1-sg2','SOAT1-sg3')
for(sgRNA_identity_select in sgRNA_identity_list){
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        ident.1 = sgRNA_identity_select,ident.2 = 'NT',
        group.by = 'sgRNA_identity',
        logfc.threshold = 0.1,
        test.use = "wilcox",
        only.pos = FALSE
    )
    gene.data <- diff_gene%>% 
      filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
      pull(avg_log2FC)
    names(gene.data) <- diff_gene %>% 
      filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
      rownames()
    gene.data %>% length() %>% message(sgRNA_identity_select,';diff Gene num: ',.)
    pv.out <- pathview(
        gene.data = gene.data, 
        pathway.id = "00100",
        species = "hsa", 
        gene.idtype = 'SYMBOL',
        out.suffix = sgRNA_identity_select,
        kegg.native = TRUE) 
    saveRDS(pv.out,paste('./Cholesterol biosynthesis_00100_',sgRNA_identity_select,'.rds',sep = ''))
    pv.out <- pathview(
        gene.data = gene.data, 
        pathway.id = "04979",
        species = "hsa", 
        gene.idtype = 'SYMBOL',
        out.suffix = sgRNA_identity_select,
        kegg.native = TRUE) 
    saveRDS(pv.out,paste('./Cholesterol metabolism_04979_',sgRNA_identity_select,'.rds',sep = ''))
}

In [ ]:
pathview_self <- function (gene.data = NULL, cpd.data = NULL, pathway.id, species = "hsa", 
          kegg.dir = ".", cpd.idtype = "kegg", gene.idtype = "entrez", 
          gene.annotpkg = NULL, min.nnodes = 3, kegg.native = TRUE, 
          map.null = TRUE, expand.node = FALSE, split.group = FALSE, 
          map.symbol = TRUE, map.cpdname = TRUE, node.sum = "sum", 
          discrete = list(gene = FALSE, cpd = FALSE), limit = list(gene = 1, 
                                                                   cpd = 1), bins = list(gene = 10, cpd = 10), both.dirs = list(gene = T, 
                                                                                                                                cpd = T), trans.fun = list(gene = NULL, cpd = NULL), 
          low = list(gene = "green", cpd = "blue"), mid = list(gene = "gray", 
                                                               cpd = "gray"), high = list(gene = "red", cpd = "yellow"), 
          na.col = "transparent", ...) 
{
  dtypes = !is.null(gene.data) + !is.null(cpd.data)
  cond0 = dtypes == 1 & is.numeric(limit) & length(limit) > 
    1
  if (cond0) {
    if (limit[1] != limit[2] & is.null(names(limit))) 
      limit = list(gene = limit[1:2], cpd = limit[1:2])
  }
  if (is.null(trans.fun)) 
    trans.fun = list(gene = NULL, cpd = NULL)
  arg.len2 = c("discrete", "limit", "bins", "both.dirs", "trans.fun", 
               "low", "mid", "high")
  for (arg in arg.len2) {
    obj1 = eval(as.name(arg))
    if (length(obj1) == 1) 
      obj1 = rep(obj1, 2)
    if (length(obj1) > 2) 
      obj1 = obj1[1:2]
    obj1 = as.list(obj1)
    ns = names(obj1)
    if (length(ns) == 0 | !all(c("gene", "cpd") %in% ns)) 
      names(obj1) = c("gene", "cpd")
    assign(arg, obj1)
  }
  if (is.character(gene.data)) {
    gd.names = gene.data
    gene.data = rep(1, length(gene.data))
    names(gene.data) = gd.names
    both.dirs$gene = FALSE
    ng = length(gene.data)
    nsamp.g = 1
  }
  else if (!is.null(gene.data)) {
    if (length(dim(gene.data)) == 2) {
      gd.names = rownames(gene.data)
      ng = nrow(gene.data)
      nsamp.g = 2
    }
    else if (is.numeric(gene.data) & is.null(dim(gene.data))) {
      gd.names = names(gene.data)
      ng = length(gene.data)
      nsamp.g = 1
    }
    else stop("wrong gene.data format!")
  }
  else if (is.null(cpd.data)) {
    stop("gene.data and cpd.data are both NULL!")
  }
  gene.idtype = toupper(gene.idtype)
  data(bods)
  if (species != "ko") {
    species.data = kegg.species.code(species, na.rm = T, 
                                     code.only = FALSE)
  }
  else {
    species.data = c(kegg.code = "ko", entrez.gnodes = "0", 
                     kegg.geneid = "K01488", ncbi.geneid = NA, ncbi.proteinid = NA, 
                     uniprot = NA)
    gene.idtype = "KEGG"
    msg.fmt = "Only KEGG ortholog gene ID is supported, make sure it looks like \"%s\"!"
    msg = sprintf(msg.fmt, species.data["kegg.geneid"])
    message("Note: ", msg)
  }
  if (length(dim(species.data)) == 2) {
    message("Note: ", "More than two valide species!")
    species.data = species.data[1, ]
  }
  species = species.data["kegg.code"]
  entrez.gnodes = species.data["entrez.gnodes"] == 1
  if (is.na(species.data["ncbi.geneid"])) {
    if (!is.na(species.data["kegg.geneid"])) {
      msg.fmt = "Mapping via KEGG gene ID (not Entrez) is supported for this species,\nit looks like \"%s\"!"
      msg = sprintf(msg.fmt, species.data["kegg.geneid"])
      message("Note: ", msg)
    }
    else {
      stop("This species is not annotated in KEGG!")
    }
  }
  if (is.null(gene.annotpkg)) 
    gene.annotpkg = bods[match(species, bods[, 3]), 1]
  if (length(grep("ENTREZ|KEGG|NCBIPROT|UNIPROT", gene.idtype)) < 
      1 & !is.null(gene.data)) {
    if (is.na(gene.annotpkg)) 
      stop("No proper gene annotation package available!")
    if (!gene.idtype %in% gene.idtype.bods[[species]]) 
      stop("Wrong input gene ID type!")
    gene.idmap = id2eg(gd.names, category = gene.idtype, 
                       pkg.name = gene.annotpkg, unique.map = F)
    gene.data = mol.sum(gene.data, gene.idmap)
    gene.idtype = "ENTREZ"
  }
  if (gene.idtype != "KEGG" & !entrez.gnodes & !is.null(gene.data)) {
    id.type = gene.idtype
    if (id.type == "ENTREZ") 
      id.type = "ENTREZID"
    kid.map = names(species.data)[-c(1:2)]
    kid.types = names(kid.map) = c("KEGG", "ENTREZID", "NCBIPROT", 
                                   "UNIPROT")
    kid.map2 = gsub("[.]", "-", kid.map)
    kid.map2["UNIPROT"] = "up"
    if (is.na(kid.map[id.type])) 
      stop("Wrong input gene ID type for the species!")
    message("Info: Getting gene ID data from KEGG...")
    gene.idmap = keggConv(kid.map2[id.type], species)
    message("Info: Done with data retrieval!")
    kegg.ids = gsub(paste(species, ":", sep = ""), "", names(gene.idmap))
    in.ids = gsub(paste0(kid.map2[id.type], ":"), "", gene.idmap)
    gene.idmap = cbind(in.ids, kegg.ids)
    gene.data = mol.sum(gene.data, gene.idmap)
    gene.idtype = "KEGG"
  }
  if (is.character(cpd.data)) {
    cpdd.names = cpd.data
    cpd.data = rep(1, length(cpd.data))
    names(cpd.data) = cpdd.names
    both.dirs$cpd = FALSE
    ncpd = length(cpd.data)
  }
  else if (!is.null(cpd.data)) {
    if (length(dim(cpd.data)) == 2) {
      cpdd.names = rownames(cpd.data)
      ncpd = nrow(cpd.data)
    }
    else if (is.numeric(cpd.data) & is.null(dim(cpd.data))) {
      cpdd.names = names(cpd.data)
      ncpd = length(cpd.data)
    }
    else stop("wrong cpd.data format!")
  }
  if (length(grep("kegg", cpd.idtype)) < 1 & !is.null(cpd.data)) {
    data(rn.list)
    cpd.types = c(names(rn.list), "name")
    cpd.types = tolower(cpd.types)
    cpd.types = cpd.types[-grep("kegg", cpd.types)]
    if (!tolower(cpd.idtype) %in% cpd.types) 
      stop("Wrong input cpd ID type!")
    cpd.idmap = cpd2kegg(cpdd.names, in.type = cpd.idtype)
    cpd.data = mol.sum(cpd.data, cpd.idmap)
  }
  warn.fmt = "Parsing %s file failed, please check the file!"
  if (length(grep(species, pathway.id)) > 0) {
    pathway.name = pathway.id
    pathway.id = gsub(species, "", pathway.id)
  }
  else pathway.name = paste(species, pathway.id, sep = "")
  kfiles = list.files(path = kegg.dir, pattern = "[.]xml|[.]png")
  npath = length(pathway.id)
  out.list = list()
  tfiles.xml = paste(pathway.name, "xml", sep = ".")
  tfiles.png = paste(pathway.name, "png", sep = ".")
  if (kegg.native) 
    ttype = c("xml", "png")
  else ttype = "xml"
  xml.file <- paste(kegg.dir, "/", tfiles.xml, sep = "")
  for (i in 1:npath) {
    if (kegg.native) 
      tfiles = c(tfiles.xml[i], tfiles.png[i])
    else tfiles = tfiles.xml[i]
    if (!all(tfiles %in% kfiles)) {
      dstatus = download.kegg(pathway.id = pathway.id[i], 
                              species = species, kegg.dir = kegg.dir, file.type = ttype)
      if (dstatus == "failed") {
        warn.fmt = "Failed to download KEGG xml/png files, %s skipped!"
        warn.msg = sprintf(warn.fmt, pathway.name[i])
        message("Warning: ", warn.msg)
        return(invisible(0))
      }
    }
    if (kegg.native) {
      node.data = try(node.info(xml.file[i]), silent = T)
      if (class(node.data)[1] == "try-error") {
        warn.msg = sprintf(warn.fmt, xml.file[i])
        message("Warning: ", warn.msg)
        return(invisible(0))
      }
      node.type = c("gene", "enzyme", "compound", "ortholog")
      sel.idx = node.data$type %in% node.type
      nna.idx = !is.na(node.data$x + node.data$y + node.data$width + 
                         node.data$height)
      sel.idx = sel.idx & nna.idx
      if (sum(sel.idx) < min.nnodes) {
        warn.fmt = "Number of mappable nodes is below %d, %s skipped!"
        warn.msg = sprintf(warn.fmt, min.nnodes, pathway.name[i])
        message("Warning: ", warn.msg)
        return(invisible(0))
      }
      node.data = lapply(node.data, "[", sel.idx)
    }
    else {
      gR1 = try(parseKGML2Graph2(xml.file[i], genes = F, 
                                 expand = expand.node, split.group = split.group), 
                silent = T)
      node.data = try(node.info(gR1), silent = T)
      if (class(node.data)[1] == "try-error") {
        warn.msg = sprintf(warn.fmt, xml.file[i])
        message("Warning: ", warn.msg)
        return(invisible(0))
      }
    }
    if (species == "ko") 
      gene.node.type = "ortholog"
    else gene.node.type = "gene"
    if ((!is.null(gene.data) | map.null) & sum(node.data$type == 
                                               gene.node.type) > 1) {
      plot.data.gene = node.map(gene.data, node.data, 
                                node.types = gene.node.type, node.sum = node.sum, 
                                entrez.gnodes = entrez.gnodes)
      kng = plot.data.gene$kegg.names
      kng.char = gsub("[0-9]", "", unlist(kng))
      if (any(kng.char > "")) 
        entrez.gnodes = FALSE
      if (map.symbol & species != "ko" & entrez.gnodes) {
        if (is.na(gene.annotpkg)) {
          warn.fmt = "No annotation package for the species %s, gene symbols not mapped!"
          warn.msg = sprintf(warn.fmt, species)
          message("Warning: ", warn.msg)
        }
        else {
          plot.data.gene$labels = eg2id(as.character(plot.data.gene$kegg.names), 
                                        category = "SYMBOL", pkg.name = gene.annotpkg)[, 
                                                                                       2]
          mapped.gnodes = rownames(plot.data.gene)
          node.data$labels[mapped.gnodes] = plot.data.gene$labels
        }
      }
      cols.ts.gene = node.color(plot.data.gene, limit$gene, 
                                bins$gene, both.dirs = both.dirs$gene, trans.fun = trans.fun$gene, 
                                discrete = discrete$gene, low = low$gene, mid = mid$gene, 
                                high = high$gene, na.col = na.col)
    }
    else plot.data.gene = cols.ts.gene = NULL
    if ((!is.null(cpd.data) | map.null) & sum(node.data$type == 
                                              "compound") > 1) {
      plot.data.cpd = node.map(cpd.data, node.data, node.types = "compound", 
                               node.sum = node.sum)
      if (map.cpdname & !kegg.native) {
        plot.data.cpd$labels = cpdkegg2name(plot.data.cpd$labels)[, 
                                                                  2]
        mapped.cnodes = rownames(plot.data.cpd)
        node.data$labels[mapped.cnodes] = plot.data.cpd$labels
      }
      cols.ts.cpd = node.color(plot.data.cpd, limit$cpd, 
                               bins$cpd, both.dirs = both.dirs$cpd, trans.fun = trans.fun$cpd, 
                               discrete = discrete$cpd, low = low$cpd, mid = mid$cpd, 
                               high = high$cpd, na.col = na.col)
    }
    else plot.data.cpd = cols.ts.cpd = NULL
    if (kegg.native) {
      pv.pars = keggview.native_self(plot.data.gene = plot.data.gene, 
                                cols.ts.gene = cols.ts.gene, plot.data.cpd = plot.data.cpd, 
                                cols.ts.cpd = cols.ts.cpd, node.data = node.data, 
                                pathway.name = pathway.name[i], kegg.dir = kegg.dir, 
                                limit = limit, bins = bins, both.dirs = both.dirs, 
                                discrete = discrete, low = low, mid = mid, high = high, 
                                na.col = na.col, ...)
    }
    else {
      pv.pars = keggview.graph(plot.data.gene = plot.data.gene, 
                               cols.ts.gene = cols.ts.gene, plot.data.cpd = plot.data.cpd, 
                               cols.ts.cpd = cols.ts.cpd, node.data = node.data, 
                               path.graph = gR1, pathway.name = pathway.name[i], 
                               map.cpdname = map.cpdname, split.group = split.group, 
                               limit = limit, bins = bins, both.dirs = both.dirs, 
                               discrete = discrete, low = low, mid = mid, high = high, 
                               na.col = na.col, ...)
    }
    plot.data.gene = cbind(plot.data.gene, cols.ts.gene)
    if (!is.null(plot.data.gene)) {
      cnames = colnames(plot.data.gene)[-(1:8)]
      nsamp = length(cnames)/2
      if (nsamp > 1) {
        cnames[(nsamp + 1):(2 * nsamp)] = paste(cnames[(nsamp + 
                                                          1):(2 * nsamp)], "col", sep = ".")
      }
      else cnames[2] = "mol.col"
      colnames(plot.data.gene)[-(1:8)] = cnames
    }
    plot.data.cpd = cbind(plot.data.cpd, cols.ts.cpd)
    if (!is.null(plot.data.cpd)) {
      cnames = colnames(plot.data.cpd)[-(1:8)]
      nsamp = length(cnames)/2
      if (nsamp > 1) {
        cnames[(nsamp + 1):(2 * nsamp)] = paste(cnames[(nsamp + 
                                                          1):(2 * nsamp)], "col", sep = ".")
      }
      else cnames[2] = "mol.col"
      colnames(plot.data.cpd)[-(1:8)] = cnames
    }
    out.list[[i]] = list(plot.data.gene = plot.data.gene, 
                         plot.data.cpd = plot.data.cpd)
  }
  if (npath == 1) 
    out.list = out.list[[1]]
  else names(out.list) = pathway.name
  return(invisible(out.list))
}

keggview.native_self <- function (plot.data.gene = NULL, plot.data.cpd = NULL, cols.ts.gene = NULL, 
                             cols.ts.cpd = NULL, node.data, pathway.name, out.suffix = "pathview", 
                             kegg.dir = ".", multi.state = TRUE, match.data = TRUE, same.layer = TRUE, 
                             res = 300, cex = 0.25, discrete = list(gene = FALSE, cpd = FALSE), 
                             limit = list(gene = 1, cpd = 1), bins = list(gene = 10, 
                                                                          cpd = 10), both.dirs = list(gene = T, cpd = T), low = list(gene = "green", 
                                                                                                                                     cpd = "blue"), mid = list(gene = "gray", cpd = "gray"), 
                             high = list(gene = "red", cpd = "yellow"), na.col = "transparent", 
                             new.signature = TRUE, plot.col.key = TRUE, key.align = "x", 
                             key.pos = "topright", ...) 
{
  img <- png::readPNG(paste(kegg.dir, "/", pathway.name, ".png",sep = ""))
  width <- ncol(img)
  height <- nrow(img)
  message('width:',width,";height:",height)
  cols.ts.gene = cbind(cols.ts.gene)
  cols.ts.cpd = cbind(cols.ts.cpd)
  nc.gene = max(ncol(cols.ts.gene), 0)
  nc.cpd = max(ncol(cols.ts.cpd), 0)
  nplots = max(nc.gene, nc.cpd)
  pn.suffix = colnames(cols.ts.gene)
  if (length(pn.suffix) < nc.cpd) 
    pn.suffix = colnames(cols.ts.cpd)
  if (length(pn.suffix) < nplots) 
    pn.suffix = 1:nplots
  if (length(pn.suffix) == 1) {
    pn.suffix = out.suffix
  }
  else pn.suffix = paste(out.suffix, pn.suffix, sep = ".")
  na.col = pathview:::colorpanel2(1, low = na.col, high = na.col)
  if ((match.data | !multi.state) & nc.gene != nc.cpd) {
    if (nc.gene > nc.cpd & !is.null(cols.ts.cpd)) {
      na.mat = matrix(na.col, ncol = nplots - nc.cpd, 
                      nrow = nrow(cols.ts.cpd))
      cols.ts.cpd = cbind(cols.ts.cpd, na.mat)
    }
    if (nc.gene < nc.cpd & !is.null(cols.ts.gene)) {
      na.mat = matrix(na.col, ncol = nplots - nc.gene, 
                      nrow = nrow(cols.ts.gene))
      cols.ts.gene = cbind(cols.ts.gene, na.mat)
    }
    nc.gene = nc.cpd = nplots
  }
  out.fmt = "Working in directory %s"
  wdir = getwd()
  out.msg = sprintf(out.fmt, wdir)
  message("Info: ", out.msg)
  out.fmt = "Writing image file %s"
  multi.state = multi.state & nplots > 1
  if (multi.state) {
    nplots = 1
    pn.suffix = paste(out.suffix, "multi", sep = ".")
    if (nc.gene > 0) 
      cols.gene.plot = cols.ts.gene
    if (nc.cpd > 0) 
      cols.cpd.plot = cols.ts.cpd
  }
  for (np in 1:nplots) {
    img.file = paste(pathway.name, pn.suffix[np], "pdf", 
                     sep = ".")
    out.msg = sprintf(out.fmt, img.file)
    message("Info: ", out.msg)
    # png(img.file, width = width, height = height, res = res)
    pdf(img.file, width = width/100*3, height = height/100*3)
    op = par(mar = c(0, 0, 0, 0))
    plot(c(0, width), c(0, height), type = "n", xlab = "", 
         ylab = "", xaxs = "i", yaxs = "i")
    if (new.signature) 
      img[height - 4:25, 17:137, 1:3] = 1
    if (same.layer != T) 
      rasterImage(img, 0, 0, width, height, interpolate = F)
    if (!is.null(cols.ts.gene) & nc.gene >= np) {
      if (!multi.state) 
        cols.gene.plot = cols.ts.gene[, np]
      if (same.layer != T) {
        pathview:::render.kegg.node(plot.data.gene, cols.gene.plot, 
                         img, same.layer = same.layer, type = "gene", 
                         cex = cex)
      }
      else {
        img = pathview:::render.kegg.node(plot.data.gene, cols.gene.plot, 
                               img, same.layer = same.layer, type = "gene")
      }
    }
    if (!is.null(cols.ts.cpd) & nc.cpd >= np) {
      if (!multi.state) 
        cols.cpd.plot = cols.ts.cpd[, np]
      if (same.layer != T) {
        pathview:::render.kegg.node(plot.data.cpd, cols.cpd.plot, 
                         img, same.layer = same.layer, type = "compound", 
                         cex = cex)
      }
      else {
        img = pathview:::render.kegg.node(plot.data.cpd, cols.cpd.plot, 
                               img, same.layer = same.layer, type = "compound")
      }
    }
    if (same.layer == T) 
      rasterImage(img, 0, 0, width, height, interpolate = F)
    pv.pars = list()
    pv.pars$gsizes = c(width = width, height = height)
    pv.pars$nsizes = c(46, 17)
    pv.pars$op = op
    pv.pars$key.cex = 2 * 72/res
    pv.pars$key.lwd = 1.2 * 72/res
    pv.pars$sign.cex = cex
    off.sets = c(x = 0, y = 0)
    align = "n"
    ucol.gene = unique(as.vector(cols.ts.gene))
    na.col.gene = ucol.gene %in% c(na.col, NA)
    if (plot.col.key & !is.null(cols.ts.gene) & !all(na.col.gene)) {
      off.sets = col.key(limit = limit$gene, bins = bins$gene, 
                         both.dirs = both.dirs$gene, discrete = discrete$gene, 
                         graph.size = pv.pars$gsizes, node.size = pv.pars$nsizes, 
                         key.pos = key.pos, cex = pv.pars$key.cex, lwd = pv.pars$key.lwd, 
                         low = low$gene, mid = mid$gene, high = high$gene, 
                         align = "n")
      align = key.align
    }
    ucol.cpd = unique(as.vector(cols.ts.cpd))
    na.col.cpd = ucol.cpd %in% c(na.col, NA)
    if (plot.col.key & !is.null(cols.ts.cpd) & !all(na.col.cpd)) {
      off.sets = col.key(limit = limit$cpd, bins = bins$cpd, 
                         both.dirs = both.dirs$cpd, discrete = discrete$cpd, 
                         graph.size = pv.pars$gsizes, node.size = pv.pars$nsizes, 
                         key.pos = key.pos, off.sets = off.sets, cex = pv.pars$key.cex, 
                         lwd = pv.pars$key.lwd, low = low$cpd, mid = mid$cpd, 
                         high = high$cpd, align = align)
    }
    if (new.signature) 
      pathview:::pathview.stamp(x = 17, y = 20, on.kegg = T, cex = pv.pars$sign.cex)
    par(pv.pars$op)
    dev.off()
  }
  return(invisible(pv.pars))
}

In [ ]:
getwd()
seurat_obj_use@meta.data %>% colnames()
seurat_obj_use$sgRNA_type_use %>% table()

In [ ]:
sgRNA_identity_list <- c('LDLR','SOAT1')
for(sgRNA_identity_select in sgRNA_identity_list){
    diff_gene <- FindMarkers(
        object = seurat_obj_use,
        ident.1 = sgRNA_identity_select,ident.2 = 'NT',
        group.by = 'sgRNA_type_use',
        logfc.threshold = 0.1,
        test.use = "wilcox",
        only.pos = FALSE
    )
    gene.data <- diff_gene%>% 
      filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
      pull(avg_log2FC)
    names(gene.data) <- diff_gene %>% 
      filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
      rownames()
    gene.data %>% length() %>% message(sgRNA_identity_select,';diff Gene num: ',.)
    pv.out <- pathview_self(
        gene.data = gene.data, 
        pathway.id = "00100",
        species = "hsa", 
        gene.idtype = 'SYMBOL',
        out.suffix = sgRNA_identity_select,
        kegg.native = TRUE) 
    saveRDS(pv.out,paste('./Cholesterol biosynthesis_00100_',sgRNA_identity_select,'.rds',sep = ''))
    pv.out <- pathview_self(
        gene.data = gene.data, 
        pathway.id = "04979",
        species = "hsa", 
        gene.idtype = 'SYMBOL',
        out.suffix = sgRNA_identity_select,
        kegg.native = TRUE) 
    saveRDS(pv.out,paste('./Cholesterol metabolism_04979_',sgRNA_identity_select,'.rds',sep = ''))
}

## endline